# Prepare a request of imagery at sites from a vendor using the CSDA Evaluation Sites GeoJSON

Paul Montesano, PhD  
June 2026

In [2]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

### Read the CSDA Sites GeoJSON stored on GitHub

+ This GeoJSON is built directly off the CSDA Evaluation Sites Database.  
+ The notebook to process this GeoJSON is here: https://github.com/pahbs/csda_summaries/blob/master/notebooks/csda_eval_sites_process.ipynb

In [3]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
sites_url = f'{RAW_BASE}/sites/csda_sites_aoi.geojson'
sites = gpd.read_file(sites_url)

/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3.8/site-packages/pyproj/../../.././libtiff.so.6: version `LIBTIFF_4.6.1' not found (required by /app/jupyter/ilab/jupyter-lab/prod/lib/gdalplugins/../libgdal.so.36)
/panfs/ccds02/app/modules/jupyter/ilab/tensorflow-kernel/lib/python3

### Indicate a name for the vendor

In [19]:
VENDOR_NAME = 'Tanager' # Change this
VENDOR_NAME = 'satellogic_v2' # Change this

In [20]:
# Get today's date
DATE = datetime.now().strftime('%Y%m%d')
DATE

'20260701'

In [21]:
#sites.info()

### Check some useful site attributes

In [22]:
print(list(sites['Evaluation Category'].unique()))

['Geometric', 'Radiometric', 'Radiometric & Geometric', 'InSAR', 'Geometric (vert/horiz)']


### Each site's 'Site Name' is the key identifier for indicating the location of a CSDA request of data from a vendor

In [23]:
print(list(sites['Site Name'].unique()))

['Baoshan', 'Cairns', 'Copiapo', 'Luanda', 'Manta', 'Mexico City', 'Mogadishu', 'Perth', 'Albuquerque', 'Amazon', 'Baotou', 'Atacama Desert', 'Belo Horizonte', 'Boston', 'Kansas City', 'San Diego', 'Los Angeles', 'Novosibirsk', 'Konya-airport', 'Konya-city', 'Mansoura', 'Cape Town', 'Casablanca', 'Caspian Sea', 'Catania', 'Crater Lake', 'Cuprite', 'Antarctica GPS', 'Doldrums', 'Atlantic Doldrums', 'Dublin', 'Gobabeb', 'Petermann Glacier', 'NISAR CR Array', 'Hohhot', 'Tianjin Docks', 'San Mateo Bridge', 'King Fahd Causeway', 'La Crau', 'Lake Pontchartrain Causeway', 'London', 'Melbourne', 'Navarre Causeway', 'DLR CR Array', 'Arabian Peninsula', 'Old Bahia Bridge', 'Phoenix', 'PICS Algeria-3', 'PICS Libya-1', 'PICS Libya-4', 'Piedmont', 'Railroad Valley', 'Buenos Aires', 'Rio Gallegos', 'RCRA', 'Salon-de-Provence', 'Sapporo', 'Suramadu Bridge', 'Shadnagar', 'Singapore', 'Sioux Falls', 'FMI CR Array', 'Valencia', 'Golmud', 'OPERA CR Array', 'WLEF', 'Etang de Berre', 'La Crau TIR', 'Lagere

#### Function to update site attributes

In [24]:
def update_sites_attributes(sites_gdf, site_configs):
    """
    Update sites GeoDataFrame with attributes based on configuration.
    
    Parameters:
    -----------
    sites_gdf : GeoDataFrame
        Sites geodataframe to update
    site_configs : list of dict
        List of configurations, each with 'sites' and 'order_parameters' keys
        
    Returns:
    --------
    GeoDataFrame : Updated sites (copy)
    list : All site names from configs
    """
    sites_updated = sites_gdf.copy()
    all_sites = []
    
    for config in site_configs:
        site_list = config['sites']
        attributes = config['order_parameters']
        
        # Update attributes for these sites
        mask = sites_updated['Site Name'].isin(site_list)
        for key, value in attributes.items():
            sites_updated.loc[mask, key] = value
        
        all_sites.extend(site_list)
    
    return sites_updated, all_sites

def print_site_update_report(sites_original, sites_updated, site_configs):
    """
    Print a report showing what attributes were updated for which sites.
    
    Parameters:
    -----------
    sites_original : GeoDataFrame
        Original sites before updates
    sites_updated : GeoDataFrame
        Sites after updates
    site_configs : list of dict
        Configuration used for updates
    """
    print("=" * 70)
    print("SITE ATTRIBUTE UPDATE REPORT")
    print("=" * 70)
    
    # Get all unique attributes being updated
    all_attributes = set()
    for config in site_configs:
        all_attributes.update(config['order_parameters'].keys())
    
    total_sites = 0
    
    for i, config in enumerate(site_configs, 1):
        sites_list = config['sites']
        attributes = config['order_parameters']
        
        print(f"\nGroup {i}: {len(sites_list)} site(s)")
        print("-" * 70)
        
        for site in sites_list:
            total_sites += 1
            print(f"\n  Site: {site}")
            
            # Get before/after values
            orig_row = sites_original[sites_original['Site Name'] == site]
            updated_row = sites_updated[sites_updated['Site Name'] == site]
            
            if len(orig_row) == 0:
                print(f"    ⚠️  WARNING: Site not found in original dataframe")
                continue
            
            for attr, new_value in attributes.items():
                old_value = orig_row[attr].iloc[0] if attr in orig_row.columns else 'N/A'
                actual_value = updated_row[attr].iloc[0] if len(updated_row) > 0 else 'ERROR'
                
                # Check if update was successful
                if str(actual_value) == str(new_value):
                    status = "✓"
                else:
                    status = "✗"
                
                print(f"    {status} {attr:20s}: {old_value} → {new_value}")
    
    print("\n" + "=" * 70)
    print(f"Total sites updated: {total_sites}")
    print("=" * 70)


### Update config of request parameters for sites chosen for this vendor request

Here is where we config & specify our 'timeseries' sites and any other types of sites we need to config & specify for this request

In [25]:
# This SITE_CONFIGS dictionary gives us our final list of sites and their order parameters for this request
# SITE_CONFIGS = [
#     {
#         'sites': ['Albuquerque', 'Casablanca'],
#         'order_parameters': {
#             'ideal_num_acqs': 10,
#             'request_type': 'timeseries',
#             'assessment_domain': 'geometric'
#         }
#     },
#     # Just examples of other configs
#     {
#         'sites': ['Baotou'],
#         'order_parameters': {
#             'ideal_num_acqs': 5,
#             'request_type': 'other',
#             'assessment_domain': 'geometric'
#         }
#     },
#     {
#         'sites': ['WLEF', 'PICS Libya-4'],
#         'order_parameters': {
#             'ideal_num_acqs': 3,
#             'request_type': 'other',
#             'assessment_domain': 'radiometric'
#         }
#     }
# ]

SITE_CONFIGS = [
    {
        "sites": ["PICS Libya-4"],
        "order_parameters": {
            "location_name": "PICS Libya-4",
            "country": "Libya",
            "longitude": 23.39,
            "latitude": 28.55,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "custom",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 5,
            "ideal_num_acqs": 5
        }
    },
    {
        "sites": ["Baotou"],
        "order_parameters": {
            "location_name": "China cal/val",
            "country": "China",
            "longitude": 109.629437,
            "latitude": 40.851787,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric & Geometric",
            "assessment_types": "Radiometric, Geometric Calibration Quality (LSF/MTF)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 0,
            "aoi_size_km": 0.04,
            "max_view_angle": 15,
            "min_num_acqs": 1,
            "ideal_num_acqs": 2
        }
    },
    {
        "sites": ["Rio Gallegos"],
        "order_parameters": {
            "location_name": "Argentina",
            "country": "Argentina",
            "longitude": -69.242713,
            "latitude": -51.625811,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Geometric",
            "assessment_types": "Geometric Calibration Quality (priority pointing site)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 15,
            "aoi_size_km": 3.0,
            "max_view_angle": 30,
            "min_num_acqs": 5,
            "ideal_num_acqs": 5
        }
    },
    {
        "sites": ["Albuquerque"],
        "order_parameters": {
            "location_name": "New Mexico",
            "country": "USA",
            "longitude": -106.613826,
            "latitude": 35.068706,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Geometric",
            "assessment_types": "Geometric Calibration Quality (priority pointing site, TS)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 15,
            "aoi_size_km": 3.0,
            "max_view_angle": 30,
            "min_num_acqs": 10,
            "ideal_num_acqs": 10
        }
    },
    {
        "sites": ["Catania"],
        "order_parameters": {
            "location_name": "Sicily",
            "country": "Italy",
            "longitude": 15.0654,
            "latitude": 37.4699,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Geometric",
            "assessment_types": "Geometric Calibration Quality (pointing site )",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 15,
            "aoi_size_km": 3.0,
            "max_view_angle": 30,
            "min_num_acqs": 5,
            "ideal_num_acqs": 5
        }
    },
    {
        "sites": ["Casablanca"],
        "order_parameters": {
            "location_name": "Casablanca",
            "country": "Morocco",
            "longitude": -7.62242,
            "latitude": 33.58037,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Geometric",
            "assessment_types": "Geometric Calibration Quality (priority pointing site)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 15,
            "aoi_size_km": 3.0,
            "max_view_angle": 30,
            "min_num_acqs": 5,
            "ideal_num_acqs": 5
        }
    },
    {
        "sites": ["Singapore"],
        "order_parameters": {
            "location_name": "Singapore",
            "country": "Singapore",
            "longitude": 103.838974,
            "latitude": 1.308939,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Geometric",
            "assessment_types": "Geometric Calibration Quality (priority pointing site)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 15,
            "aoi_size_km": 3.0,
            "max_view_angle": 30,
            "min_num_acqs": 5,
            "ideal_num_acqs": 5
        }
    },
    {
        "sites": ["Valencia"],
        "order_parameters": {
            "location_name": "ESP-VLC2, El Palmar",
            "country": "Spain",
            "longitude": -0.3263,
            "latitude": 39.2979,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "circle",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 1,
            "ideal_num_acqs": 1
        }
    },
    {
        "sites": ["WLEF"],
        "order_parameters": {
            "location_name": "WLEF tower",
            "country": "USA",
            "longitude": -90.2732,
            "latitude": 45.9449,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "circle",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 1,
            "ideal_num_acqs": 1
        }
    },
    {
        "sites": ["Gobabeb"],
        "order_parameters": {
            "location_name": "Gobabeb",
            "country": "Namibia",
            "longitude": 15.11956,
            "latitude": -23.6002,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 5,
            "ideal_num_acqs": 10
        }
    },
    {
        "sites": ["Railroad Valley"],
        "order_parameters": {
            "location_name": "Nevada",
            "country": "USA",
            "longitude": -115.69,
            "latitude": 38.497,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 5,
            "ideal_num_acqs": 10
        }
    },
    {
        "sites": ["Golmud"],
        "order_parameters": {
            "location_name": "Golmud",
            "country": "China",
            "longitude": 94.3286,
            "latitude": 36.3977,
            "remote_sensing_domain": "Optical Multi/Hyper",
            "evaluation_category": "Radiometric",
            "assessment_types": "Radiometric Calibration Quality (Absolute)",
            "aoi_shape": "box",
            "max_aoi_cloud_pct": 0,
            "max_scene_cloud_pct": 20,
            "aoi_size_km": 3.0,
            "max_view_angle": 10,
            "min_num_acqs": 1,
            "ideal_num_acqs": 5
        }
    }
]


sites_updated, SITES_FOR_REQUEST = update_sites_attributes(sites, SITE_CONFIGS)

print_site_update_report(sites, sites_updated, SITE_CONFIGS)

SITE ATTRIBUTE UPDATE REPORT

Group 1: 1 site(s)
----------------------------------------------------------------------

  Site: PICS Libya-4
    ✓ location_name       : N/A → PICS Libya-4
    ✓ country             : N/A → Libya
    ✓ longitude           : N/A → 23.39
    ✓ latitude            : N/A → 28.55
    ✓ remote_sensing_domain: N/A → Optical Multi/Hyper
    ✓ evaluation_category : N/A → Radiometric
    ✓ assessment_types    : N/A → Radiometric Calibration Quality (Absolute)
    ✓ aoi_shape           : custom → custom
    ✗ max_aoi_cloud_pct   : N/A → 0
    ✗ max_scene_cloud_pct : N/A → 20
    ✓ aoi_size_km         : nan → 3.0
    ✗ max_view_angle      : nan → 10
    ✗ min_num_acqs        : nan → 5
    ✗ ideal_num_acqs      : nan → 5

Group 2: 1 site(s)
----------------------------------------------------------------------

  Site: Baotou
    ✓ location_name       : N/A → China cal/val
    ✓ country             : N/A → China
    ✓ longitude           : N/A → 109.629437
    ✓ lat

### Create and write subset GeoJSON for this request

In [26]:
sites_subset = sites_updated[sites_updated['Site Name'].isin(SITES_FOR_REQUEST)]
sites_subset

,site_id,Site Name abbrev,Site Name,Location Name,Country,Program Use,Longitude,Latitude,Remote Sensing Domain,Priority Level,...,geometry,location_name,country,longitude,latitude,remote_sensing_domain,evaluation_category,assessment_types,max_aoi_cloud_pct,max_scene_cloud_pct
8,1.0,Albuquerque,Albuquerque,New Mexico,USA,CSDA,-106.613826,35.068706,Optical Multi/Hyper,high,...,"POLYGON ((-106.59712 35.05540, -106.59764 35.0...",New Mexico,USA,-106.613826,35.068706,Optical Multi/Hyper,Geometric,Geometric Calibration Quality (priority pointi...,0.0,15.0
10,NaN,Baotou,Baotou,China cal/val,China,CSDA,109.629437,40.851787,Optical Multi/Hyper,high,...,"POLYGON ((109.62968 40.85161, 109.62967 40.851...",China cal/val,China,109.629437,40.851787,Optical Multi/Hyper,Radiometric & Geometric,"Radiometric, Geometric Calibration Quality (LS...",0.0,0.0
22,NaN,Casablanca,Casablanca,Casablanca,Morocco,CSDA,-7.622420,33.580370,Optical Multi/Hyper,high,...,"POLYGON ((-7.60648 33.56666, -7.60604 33.59371...",Casablanca,Morocco,-7.622420,33.580370,Optical Multi/Hyper,Geometric,Geometric Calibration Quality (priority pointi...,0.0,15.0
24,NaN,Catania,Catania,Sicily,Italy,CSDA,15.065400,37.469900,Optical Multi/Hyper,high,...,"POLYGON ((15.08235 37.45637, 15.08238 37.48341...",Sicily,Italy,15.065400,37.469900,Optical Multi/Hyper,Geometric,Geometric Calibration Quality (pointing site ),0.0,15.0
31,NaN,Gobabeb,Gobabeb,Gobabeb,Namibia,CSDA,15.119560,-23.600200,Optical Multi/Hyper,high,...,"POLYGON ((15.14897 -23.60017, 15.14883 -23.602...",Gobabeb,Namibia,15.119560,-23.600200,Optical Multi/Hyper,Radiometric,Radiometric Calibration Quality (Absolute),0.0,20.0
49,NaN,PICS Libya-4,PICS Libya-4,PICS Libya-4,Libya,CSDA,23.390000,28.550000,Optical Multi/Hyper,high,...,"POLYGON ((23.39665 28.55833, 23.39110 28.53333...",PICS Libya-4,Libya,23.390000,28.550000,Optical Multi/Hyper,Radiometric,Radiometric Calibration Quality (Absolute),0.0,20.0
51,NaN,Railroad Valley,Railroad Valley,Nevada,USA,CSDA,-115.690000,38.497000,Optical Multi/Hyper,high,...,"POLYGON ((-115.51804 38.49495, -115.51914 38.4...",Nevada,USA,-115.690000,38.497000,Optical Multi/Hyper,Radiometric,Radiometric Calibration Quality (Absolute),0.0,20.0
53,NaN,Rio Gallegos,Rio Gallegos,Argentina,Argentina,CSDA,-69.242713,-51.625811,Optical Multi/Hyper,high,...,"POLYGON ((-69.22111 -51.63934, -69.22098 -51.6...",Argentina,Argentina,-69.242713,-51.625811,Optical Multi/Hyper,Geometric,Geometric Calibration Quality (priority pointi...,0.0,15.0
59,NaN,Singapore,Singapore,Singapore,Singapore,CSDA,103.838974,1.308939,Optical Multi/Hyper/Thermal IR,high,...,"POLYGON ((103.85245 1.29537, 103.85245 1.32250...",Singapore,Singapore,103.838974,1.308939,Optical Multi/Hyper,Geometric,Geometric Calibration Quality (priority pointi...,0.0,15.0
62,NaN,Valencia,Valencia,"ESP-VLC2, El Palmar",Spain,CSDA,-0.326300,39.297900,Optical Multi/Hyper,high,...,"POLYGON ((-0.29155 39.29710, -0.29182 39.29445...","ESP-VLC2, El Palmar",Spain,-0.326300,39.297900,Optical Multi/Hyper,Radiometric,Radiometric Calibration Quality (Absolute),0.0,20.0


In [27]:
OUTPUT_DIR = '/home/pmontesa/code/csda_summaries/sites' # Specify your output dir here

In [28]:
sites_subset.to_file(f'{OUTPUT_DIR}/csda_sites_aoi_{VENDOR_NAME}_{DATE}.geojson')